# Small puzzles: base Qwen2.5-1.5B vs the run-04 adapter

The medium set is where the runs were scored, but it cannot tell you how much of the
policy is **training** versus **the base model plus the action menu**. This notebook
isolates that on the easiest distribution: the 32 small puzzles of the deterministic
dev holdout (1762 cells, 6x6 to 9x9).

Four arms, identical prompt, menu, budget and decoding:

| Arm | Weights | Branch filter |
| --- | --- | --- |
| `baseline-depth0` | base `Qwen/Qwen2.5-1.5B-Instruct`, no adapter | off |
| `baseline-depth2` | base, no adapter | depth 2 |
| `run04-depth0` | run-04 adapter | off |
| `run04-depth2` | run-04 adapter | depth 2 |

That gives two comparisons for the price of one:

* **adapter effect** — `baseline-depth0` → `run04-depth0`. Everything here is learned
  behaviour: the JSON action format, the menu discipline, the deductions.
* **filter effect** — within each pair, `depth0` → `depth2`. The search is sound, so it
  can only replace a branch value that propagation refutes; it cannot teach format.

Reference point, recorded earlier with a *different* adapter (run-03) at depth 0 /
256 actions: **31/32 solved, 1733/1762 cells, 0 invalid actions, ~122 actions per
puzzle**. Run-04 should land near that; the base model should not.

**Expected wall time:** ~30–45 min. The base arms fail fast (a reply that does not map
to the menu ends the puzzle), the run-04 arms are the ones that run to 122 actions.

**Inputs** (both datasets):

* `run-04-branch-search-depth` — the 7 adapter files, flat;
* your data dataset containing `train-puzzles-small.jsonl`.


## 1. Environment

Two `Tesla T4` -> fp16.


In [ ]:
import subprocess, sys, torch

print(sys.version)
print("torch", torch.__version__, "| cuda", torch.version.cuda)
print("gpus", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(" ", i, torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i))
if torch.cuda.is_available():
    major = torch.cuda.get_device_capability(0)[0]
    print("selected dtype:", "bfloat16" if major >= 8 else "float16 (T4 is sm_75)")
print()
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)


## 2. Dependencies

Only the evaluator is needed: no `trl`, no PEFT training stack beyond loading LoRA.
`torchao` is still removed because PEFT refuses to inject an adapter when Kaggle's
0.10 build is importable.


In [ ]:
%pip install -q "transformers==5.22.2" "trl>=1.15.0,<2" "peft>=0.21.2,<1" "datasets>=4.8.4,<5" "accelerate>=1.15.0,<2" "huggingface-hub>=1.31.0,<2" "safetensors>=0.8.0,<1"
%pip uninstall -y -q torchao


In [ ]:
import importlib.metadata as metadata

for package in ("torch", "transformers", "trl", "peft", "accelerate", "datasets"):
    print(f"{package:14s}", metadata.version(package))


## 3. Locate the adapter and the small-puzzle file

Discovery is by filename. The adapter is preferred by dataset name so an older
adapter left attached does not silently become the comparison.


In [ ]:
from pathlib import Path
import json

INPUT = Path("/kaggle/input")
ADAPTER_OVERRIDE = None   # e.g. Path("/kaggle/input/run-04-branch-search-depth")
DATA_OVERRIDE = None      # e.g. Path("/kaggle/input/train-tracks-data")

adapter_hits = sorted({hit.parent for hit in INPUT.glob("**/adapter_config.json")})
print("adapter_config.json found in:")
for hit in adapter_hits:
    print("  ", hit, "| weights:", (hit / "adapter_model.safetensors").exists())
assert adapter_hits, (
    "No adapter_config.json under /kaggle/input. Upload the 7 run-04 adapter files as "
    "a dataset (not the zip) and attach it.")
preferred = [hit for hit in adapter_hits if "run-04" in hit.name.lower()]
ADAPTER = (Path(ADAPTER_OVERRIDE) if ADAPTER_OVERRIDE
           else (preferred[0] if preferred else adapter_hits[0]))
assert (ADAPTER / "adapter_model.safetensors").exists(), (
    f"{ADAPTER} has a config but no weights")
config = json.loads((ADAPTER / "adapter_config.json").read_text())
print("\nadapter:", ADAPTER)
print("  base model:", config.get("base_model_name_or_path"),
      "| r/alpha:", config.get("r"), "/", config.get("lora_alpha"))
assert config.get("r") == 16, "expected the run-04 adapter (r=16)"

data_hits = sorted({hit.parent for hit in INPUT.glob("**/train-puzzles-small.jsonl")})
print("\ntrain-puzzles-small.jsonl found in:")
for hit in data_hits:
    print("  ", hit)
assert data_hits, (
    "No train-puzzles-small.jsonl under /kaggle/input. Attach the data dataset, or "
    "set DATA_OVERRIDE.")
DATA = Path(DATA_OVERRIDE) if DATA_OVERRIDE else data_hits[0]
print("\nusing DATA =", DATA)

MODEL = config.get("base_model_name_or_path") or "Qwen/Qwen2.5-1.5B-Instruct"
print("base model:", MODEL)


## 4. Write the evaluator

Three modules: rules/prompt/validator, branch search plus the data builder, and the
rollout evaluator. `tt_eval.py` now takes `--base-model` for the zero-shot arms and
`--adapter` for the trained ones.


In [ ]:
from pathlib import Path

TT = Path("/kaggle/working/tt")
TT.mkdir(parents=True, exist_ok=True)
print(TT)


In [ ]:
%%writefile /kaggle/working/tt/tt_core.py
# Generated from the verified Kaggle package. Edit this cell, not a file.
"""Small, inspectable constraint engine for offline Train Tracks traces.

Rows and columns are zero-based internally. Masks use the tile order in TILES.
Every deduction removes only candidates ruled out by a visible local constraint.
"""

from __future__ import annotations

from dataclasses import dataclass

TILES = ("XX", "NS", "EW", "NE", "NW", "SE", "SW")
BIT = {name: 1 << index for index, name in enumerate(TILES)}
ALL = (1 << len(TILES)) - 1
TRACK = ALL ^ BIT["XX"]
DIRECTIONS = {"N": (-1, 0, "S"), "E": (0, 1, "W"),
              "S": (1, 0, "N"), "W": (0, -1, "E")}


def choices(mask: int) -> list[str]:
    return [name for name in TILES if mask & BIT[name]]


def one(mask: int) -> str | None:
    return TILES[mask.bit_length() - 1] if mask and mask.bit_count() == 1 else None


def clue_from_record(record: dict) -> dict:
    puzzle = record["puzzle"]
    height = len(puzzle["rows"])
    givens = [(height - 1 - row, column, shape)
              for shape, row, column, *_ in puzzle["pieces"]]
    return {"rows": puzzle["rows"], "columns": puzzle["columns"],
            "given_shapes": sorted(givens)}


def initial_domains(clue: dict) -> list[list[int]]:
    height, width = len(clue["rows"]), len(clue["columns"])
    given = {(r, c): shape for r, c, shape in clue["given_shapes"]}
    board = []
    for r in range(height):
        line = []
        for c in range(width):
            if (r, c) in given:
                line.append(BIT[given[r, c]])
                continue
            allowed = 0
            for name in TILES:
                if all(0 <= r + DIRECTIONS[d][0] < height
                       and 0 <= c + DIRECTIONS[d][1] < width for d in name if d != "X"):
                    allowed |= BIT[name]
            line.append(allowed)
        board.append(line)
    return board


def snapshot(board: list[list[int]]) -> list[list[int]]:
    return [line.copy() for line in board]


@dataclass(frozen=True)
class Step:
    kind: str  # deduce or contradiction
    rule: str
    anchor: dict
    cell: tuple[int, int] | None = None
    mask: int | None = None

    def action(self) -> dict:
        result = {"type": self.kind, "rule": self.rule, "anchor": self.anchor}
        if self.cell is not None:
            result["cell"] = {"row": self.cell[0] + 1,
                              "column": self.cell[1] + 1}
        if self.mask is not None:
            result["candidates"] = choices(self.mask)
        return result


def _line_step(board, cells, clue, kind, index):
    fixed = sum(not (board[r][c] & BIT["XX"]) for r, c in cells)
    possible = sum(bool(board[r][c] & TRACK) for r, c in cells)
    anchor = {kind: index + 1, "clue": clue}
    if fixed > clue or possible < clue:
        return Step("contradiction", "clue_bounds", anchor)
    for r, c in cells:
        old = board[r][c]
        if fixed == clue and old & BIT["XX"] and old & TRACK:
            return Step("deduce", "clue_satisfied", anchor,
                        (r, c), BIT["XX"])
        if possible == clue and old & BIT["XX"] and old & TRACK:
            return Step("deduce", "all_remaining_cells_required", anchor,
                        (r, c), old & TRACK)
    return None


def _port_step(board, r, c):
    height, width = len(board), len(board[0])
    old = board[r][c]
    supported = 0
    for shape in choices(old):
        if shape == "XX":
            ports = ""
        else:
            ports = shape
        valid = True
        for direction, (dr, dc, opposite) in DIRECTIONS.items():
            nr, nc = r + dr, c + dc
            if not (0 <= nr < height and 0 <= nc < width):
                continue  # The boundary and endpoint rule is set initially.
            if not any((direction in ports) == (opposite in neighbor)
                       for neighbor in choices(board[nr][nc])):
                valid = False
                break
        if valid:
            supported |= BIT[shape]
    if supported == old:
        return None
    anchor = {"row": r + 1, "column": c + 1}
    return (Step("deduce", "port_reciprocity", anchor, (r, c), supported)
            if supported else Step("contradiction", "port_reciprocity", anchor,
                                   (r, c)))


def next_step(board: list[list[int]], clue: dict) -> Step | None:
    height, width = len(board), len(board[0])
    for r in range(height):
        event = _line_step(board, [(r, c) for c in range(width)],
                           clue["rows"][r], "row", r)
        if event:
            return event
    for c in range(width):
        event = _line_step(board, [(r, c) for r in range(height)],
                           clue["columns"][c], "column", c)
        if event:
            return event
    for r in range(height):
        for c in range(width):
            event = _port_step(board, r, c)
            if event:
                return event
    if all(one(mask) is not None for line in board for mask in line):
        error = completed_error(board, clue)
        if error:
            return Step("contradiction", error, {"board": "complete"})
    return None


def completed_error(board: list[list[int]], clue: dict) -> str | None:
    height, width = len(board), len(board[0])
    grid = [[one(mask) for mask in line] for line in board]
    if any(tile is None for line in grid for tile in line):
        return "incomplete"
    if any(sum(tile != "XX" for tile in grid[r]) != clue["rows"][r]
           for r in range(height)) or any(
               sum(grid[r][c] != "XX" for r in range(height)) != clue["columns"][c]
               for c in range(width)):
        return "clue_mismatch"
    occupied = {(r, c) for r in range(height) for c in range(width)
                if grid[r][c] != "XX"}
    given = {(r, c): shape for r, c, shape in clue["given_shapes"]}
    if any(grid[r][c] != shape for (r, c), shape in given.items()):
        return "changed_given"
    external = []
    for r, c in occupied:
        for direction in grid[r][c]:
            dr, dc, opposite = DIRECTIONS[direction]
            nr, nc = r + dr, c + dc
            if not (0 <= nr < height and 0 <= nc < width):
                external.append((r, c, direction))
            elif opposite not in grid[nr][nc]:
                return "port_reciprocity"
    expected_external = {(r, c, direction) for (r, c), shape in given.items()
                         for direction in shape if not (
                             0 <= r + DIRECTIONS[direction][0] < height and
                             0 <= c + DIRECTIONS[direction][1] < width)}
    if set(external) != expected_external or len(external) != 2:
        return "endpoints"
    if not occupied:
        return "empty_path"
    found, todo = set(), [next(iter(occupied))]
    while todo:
        cell = todo.pop()
        if cell in found:
            continue
        found.add(cell)
        r, c = cell
        for direction in grid[r][c]:
            dr, dc, _ = DIRECTIONS[direction]
            neighbor = r + dr, c + dc
            if neighbor in occupied and neighbor not in found:
                todo.append(neighbor)
    return None if found == occupied else "disconnected_path"


def apply_deduction(board, step: Step) -> None:
    if step.kind != "deduce" or step.cell is None or step.mask is None:
        raise ValueError("Expected a deduction")
    r, c = step.cell
    if step.mask == 0 or step.mask == board[r][c] or step.mask & board[r][c] != step.mask:
        raise ValueError("Deduction must remove at least one candidate")
    board[r][c] = step.mask


def branch_cell(board: list[list[int]]) -> tuple[int, int] | None:
    candidates = [(mask.bit_count(), r, c) for r, line in enumerate(board)
                  for c, mask in enumerate(line) if mask.bit_count() > 1]
    return min(candidates)[1:] if candidates else None


"""Shared text policy format and rule checker for interactive trace rollouts."""

import hashlib
import json

SYSTEM = ("Solve Train Tracks one action at a time. Choose only from the "
          "provided legal-action menu. Return exactly one JSON object with the "
          "selected id, for example {\"choice\":\"a00\"}. Do not invent an "
          "action or alter a listed action.")


def development_ids(puzzle_ids, count=32):
    ids = sorted(set(puzzle_ids), key=lambda x: hashlib.sha256(x.encode()).hexdigest())
    if len(ids) <= count:
        raise ValueError("Not enough puzzles for a disjoint train/development split")
    return set(ids[:count])


def _action_key(action):
    return json.dumps(action, sort_keys=True, separators=(",", ":"))


def legal_actions(state, pending=False):
    """Enumerate rule-supported actions; offer branches only when propagation stalls."""
    board = state["domains"]
    clue = {"rows": state["row_clues"], "columns": state["column_clues"],
            "given_shapes": [(g["row"] - 1, g["column"] - 1, g["value"])
                             for g in state["givens"]]}
    if pending:
        stack = state["branch_stack"]
        return ([{"type": "undo", "to_step": stack[-1]["to_step"]}]
                if stack else [])

    height, width = len(board), len(board[0])
    stack = state["branch_stack"]
    actions = []
    for row in range(height):
        cells = [(row, c) for c in range(width)]
        fixed = sum(not (board[r][c] & BIT["XX"]) for r, c in cells)
        possible = sum(bool(board[r][c] & TRACK) for r, c in cells)
        number = clue["rows"][row]
        anchor = {"row": row + 1, "clue": number}
        if stack and (fixed > number or possible < number):
            actions.append(Step("contradiction", "clue_bounds", anchor).action())
        for c in range(width):
            cell = (row, c)
            for rule in ("clue_satisfied", "all_remaining_cells_required"):
                step = _rule_step(board, clue, rule, anchor, cell)
                if step:
                    actions.append(step.action())
    for column in range(width):
        cells = [(r, column) for r in range(height)]
        fixed = sum(not (board[r][c] & BIT["XX"]) for r, c in cells)
        possible = sum(bool(board[r][c] & TRACK) for r, c in cells)
        number = clue["columns"][column]
        anchor = {"column": column + 1, "clue": number}
        if stack and (fixed > number or possible < number):
            actions.append(Step("contradiction", "clue_bounds", anchor).action())
        for r in range(height):
            cell = (r, column)
            for rule in ("clue_satisfied", "all_remaining_cells_required"):
                step = _rule_step(board, clue, rule, anchor, cell)
                if step:
                    actions.append(step.action())
    for r in range(height):
        for c in range(width):
            step = _port_step(board, r, c)
            if step and (step.kind == "deduce" or stack):
                actions.append(step.action())

    error = completed_error(board, clue)
    if error is None:
        actions.append({"type": "finish"})
    elif stack and error != "incomplete":
        actions.append(Step("contradiction", error, {"board": "complete"}).action())

    # A branch is a fallback after no visible deduction remains.
    if not actions:
        cell = branch_cell(board)
        if cell:
            r, c = cell
            for value in choices(board[r][c]):
                actions.append({"type": "branch",
                                "cell": {"row": r + 1, "column": c + 1},
                                "value": value})

    unique = {_action_key(action): action for action in actions}
    return [unique[key] for key in sorted(unique)]


def choice_id(action, actions):
    key = _action_key(action)
    for index, candidate in enumerate(actions):
        if _action_key(candidate) == key:
            return f"a{index:03d}"
    raise ValueError(f"Trace action is absent from its legal menu: {action}")


def prompt(state, actions):
    readable = dict(state)
    readable["domains"] = [[choices(mask) for mask in row]
                           for row in state["domains"]]
    readable["legal_actions"] = [
        {"id": f"a{index:03d}", "action": action}
        for index, action in enumerate(actions)]
    return ("Board state (each domains cell lists its remaining tiles; "
            "XX means empty):\n"
            + json.dumps(readable, separators=(",", ":"), ensure_ascii=False)
            + "\nChoose one listed legal action. JSON choice id only.")


def completion(action_id):
    return json.dumps({"choice": action_id}, separators=(",", ":"))


def parse_action(raw, actions):
    try:
        result = json.loads(raw.strip())
    except (ValueError, TypeError):
        return None
    if not isinstance(result, dict) or set(result) != {"choice"}:
        return None
    choice = result["choice"]
    if not isinstance(choice, str) or not choice.startswith("a"):
        return None
    try:
        index = int(choice[1:])
    except ValueError:
        return None
    return actions[index] if 0 <= index < len(actions) and choice == f"a{index:03d}" else None


def _cell(action, board):
    cell = action.get("cell")
    if not isinstance(cell, dict):
        return None
    r, c = cell.get("row"), cell.get("column")
    if type(r) is not int or type(c) is not int or not (1 <= r <= len(board)) or not (1 <= c <= len(board[0])):
        return None
    return r - 1, c - 1


def _rule_step(board, clue, rule, anchor, cell=None):
    if not isinstance(rule, str):
        return None
    h, w = len(board), len(board[0])
    if rule in {"clue_satisfied", "all_remaining_cells_required", "clue_bounds"}:
        if not isinstance(anchor, dict):
            return None
        if "row" in anchor and type(anchor["row"]) is int and 1 <= anchor["row"] <= h:
            i = anchor["row"] - 1
            cells = [(i, c) for c in range(w)]
            kind, number = "row", clue["rows"][i]
        elif "column" in anchor and type(anchor["column"]) is int and 1 <= anchor["column"] <= w:
            i = anchor["column"] - 1
            cells = [(r, i) for r in range(h)]
            kind, number = "column", clue["columns"][i]
        else:
            return None
        if anchor != {kind: i + 1, "clue": number}:
            return None
        fixed = sum(not (board[r][c] & BIT["XX"]) for r, c in cells)
        possible = sum(bool(board[r][c] & TRACK) for r, c in cells)
        if rule == "clue_bounds":
            return Step("contradiction", rule, anchor) if fixed > number or possible < number else None
        if cell not in cells:
            return None
        r, c = cell
        old = board[r][c]
        if not (old & BIT["XX"] and old & TRACK):
            return None
        if rule == "clue_satisfied" and fixed == number:
            return Step("deduce", rule, anchor, cell, BIT["XX"])
        if rule == "all_remaining_cells_required" and possible == number:
            return Step("deduce", rule, anchor, cell, old & TRACK)
    elif rule == "port_reciprocity" and isinstance(anchor, dict):
        if type(anchor.get("row")) is not int or type(anchor.get("column")) is not int:
            return None
        r, c = anchor["row"] - 1, anchor["column"] - 1
        if not (0 <= r < h and 0 <= c < w):
            return None
        if cell is not None and cell != (r, c):
            return None
        return _port_step(board, r, c)
    elif rule in {"disconnected_path", "endpoints", "empty_path", "clue_mismatch", "changed_given"}:
        if anchor == {"board": "complete"} and completed_error(board, clue) == rule:
            return Step("contradiction", rule, anchor)
    return None


def validate_action(board, clue, action, stack, pending):
    """Return (valid, reason, canonical action). No solution is consulted."""
    if not isinstance(action, dict):
        return False, "malformed", None
    kind = action.get("type")
    if not isinstance(kind, str):
        return False, "malformed", None
    if pending:
        if kind != "undo" or not stack or action.get("to_step") != stack[-1]["to_step"]:
            return False, "expected_undo", None
        return True, "ok", {"type": "undo", "to_step": stack[-1]["to_step"]}
    if kind in {"deduce", "contradiction"}:
        cell = _cell(action, board) if "cell" in action else None
        if "cell" in action and cell is None:
            return False, "invalid_cell", None
        step = _rule_step(board, clue, action.get("rule"), action.get("anchor"), cell)
        if step is None or step.kind != kind:
            return False, "unsupported_rule", None
        if kind == "deduce":
            if cell is None or step.cell != cell or action.get("candidates") != choices(step.mask):
                return False, "wrong_candidates", None
        elif step.cell is not None and step.cell != cell:
            return False, "wrong_contradiction_cell", None
        elif step.cell is None and cell is not None:
            return False, "unexpected_cell", None
        return True, "ok", step.action()
    if kind == "branch":
        cell = _cell(action, board)
        value = action.get("value")
        if cell is None or not isinstance(value, str) or value not in BIT or board[cell[0]][cell[1]].bit_count() < 2:
            return False, "invalid_branch", None
        if not board[cell[0]][cell[1]] & BIT[value]:
            return False, "branch_outside_domain", None
        if completed_error(board, clue) not in {"incomplete", None}:
            return False, "contradictory_board", None
        return True, "ok", {"type": "branch", "cell": action["cell"], "value": value}
    if kind == "finish":
        return (True, "ok", {"type": "finish"}) if completed_error(board, clue) is None else (False, "unfinished", None)
    return False, "unexpected_action", None


def split_examples(rows, max_deductions_per_puzzle=48):
    """Keep every start and rare positive action; balance sampled deductions by rule."""
    from collections import defaultdict
    grouped = defaultdict(list)
    for row in rows:
        if row.get("expert_policy_label"):
            grouped[row["puzzle_id"]].append(row)
    selected = []
    for puzzle_id, items in grouped.items():
        deductions = [r for r in items if r["action"]["type"] == "deduce"]
        rare = [r for r in items if r["action"]["type"] != "deduce"]
        starts = [r for r in items if r["step_index"] == 0]
        selected.extend(starts)

        start_steps = {r["step_index"] for r in starts}
        by_rule = defaultdict(list)
        for row in deductions:
            if row["step_index"] not in start_steps:
                by_rule[row["action"].get("rule", "unknown")].append(row)
        for rule, rule_rows in by_rule.items():
            rule_rows.sort(key=lambda r: hashlib.sha256(
                f"{puzzle_id}:{r['step_index']}:42".encode()).hexdigest())

        start_deductions = sum(r["action"]["type"] == "deduce" for r in starts)
        remaining = max(0, max_deductions_per_puzzle - start_deductions)
        rules = sorted(by_rule)
        while remaining and rules:
            active = []
            for rule in rules:
                if by_rule[rule]:
                    selected.append(by_rule[rule].pop())
                    remaining -= 1
                    if by_rule[rule]:
                        active.append(rule)
                    if remaining == 0:
                        break
            rules = active
        selected.extend(r for r in rare if r["step_index"] not in start_steps)
    selected.sort(key=lambda r: hashlib.sha256(
        f"{r['puzzle_id']}:{r['step_index']}:shuffle".encode()).hexdigest())
    return selected


In [ ]:
%%writefile /kaggle/working/tt/tt_lib.py
# Generated from the verified Kaggle package. Edit this cell, not a file.
"""Safe bounded search for Train Tracks branch values.

The evaluator offers a branch only when no deduction remains, and only two
values of one cell.  A wrong value can cost dozens of forced deductions before
a contradiction, so we spend a little CPU here to reject values that are
provably impossible under the same rule engine.

The filter is *safe*: it only rejects a value when propagation plus a bounded
search reaches a contradiction.  If the current board still contains the true
solution, the true value cannot be rejected.  When both values survive the
search, callers should fall back to the model's choice.

Validated on the run-03 medium rollouts: on boards that still contain the
reference solution, depth 2 raised branch accuracy from 68.2% to 92.4% and never
rejected the true value.  Depth 3 added nothing, so depth 2 is the default.
"""

from __future__ import annotations

from tt_core import legal_actions
from tt_core import BIT, TILES, branch_cell, choices, completed_error

DEFAULT_DEPTH = 2


def clue_from_state(state: dict) -> dict:
    return {
        "rows": state["row_clues"],
        "columns": state["column_clues"],
        "given_shapes": [
            (g["row"] - 1, g["column"] - 1, g["value"]) for g in state["givens"]
        ],
    }


def _mask(candidates) -> int:
    return sum(BIT[c] for c in candidates)


def propagate(board, state: dict, stack: list, round_limit: int = 400):
    """Apply every currently legal deduction until a fixpoint or contradiction.

    Returns ``(contradicted, board, deductions)``.  Actions that became stale
    after an earlier action in the same round are skipped rather than treated as
    contradictions.
    """
    board = [row[:] for row in board]
    deductions = 0
    for _ in range(round_limit):
        local = {
            "row_clues": state["row_clues"],
            "column_clues": state["column_clues"],
            "givens": state["givens"],
            "domains": board,
            "branch_stack": stack,
        }
        actions = legal_actions(local, pending=False)
        if any(a["type"] == "contradiction" for a in actions):
            return True, board, deductions
        deduces = [a for a in actions if a["type"] == "deduce"]
        if not deduces:
            return False, board, deductions
        applied = 0
        for action in deduces:
            r = action["cell"]["row"] - 1
            c = action["cell"]["column"] - 1
            new = _mask(action["candidates"])
            if new == board[r][c] or new & board[r][c] != new:
                continue
            board[r][c] = new
            deductions += 1
            applied += 1
        if not applied:
            return False, board, deductions
    return False, board, deductions


def possible(board, state: dict, stack: list, depth: int, node_budget: list | None = None) -> bool:
    """Return True if the board is not proven impossible within ``depth`` guesses."""
    if node_budget is not None:
        if node_budget[0] <= 0:
            return True
        node_budget[0] -= 1
    contradicted, board, _ = propagate(board, state, stack)
    if contradicted:
        return False
    clue = clue_from_state(state)
    if branch_cell(board) is None:
        return completed_error(board, clue) is None
    if depth <= 0:
        return True  # unknown: conservatively possible
    r, c = branch_cell(board)
    cell = {"row": r + 1, "column": c + 1}
    for value in choices(board[r][c]):
        child = [row[:] for row in board]
        child[r][c] = BIT[value]
        child_stack = list(stack) + [
            {"to_step": -1, "cell": cell, "value": value}
        ]
        if possible(child, state, child_stack, depth - 1, node_budget):
            return True
    return False


def surviving_values(state: dict, cell: dict, depth: int = DEFAULT_DEPTH,
                     node_budget: int = 4000) -> dict:
    """Probe both values at ``cell`` and return details about survivors."""
    r, c = cell["row"] - 1, cell["column"] - 1
    candidates = [t for t in TILES if state["domains"][r][c] & BIT[t]]
    budget = [node_budget]
    feasible = {}
    for value in candidates:
        child = [row[:] for row in state["domains"]]
        child[r][c] = BIT[value]
        child_stack = list(state["branch_stack"]) + [
            {"to_step": -1, "cell": cell, "value": value}
        ]
        feasible[value] = possible(child, state, child_stack, depth, budget)
    survivors = [v for v in candidates if feasible[v]]
    return {"candidates": candidates, "feasible": feasible, "survivors": survivors,
            "depth": depth, "nodes_left": budget[0]}


def choose_branch(state: dict, cell: dict, model_value: str,
                  depth: int = DEFAULT_DEPTH, node_budget: int = 4000) -> dict:
    """Pick a branch value, using the model choice when it survives.

    Returns a dict with the chosen ``value``, whether the search overrode the
    model, and the survivor details for logging.
    """
    if depth <= 0:
        return {"value": model_value, "override": False, "search": None}
    info = surviving_values(state, cell, depth=depth, node_budget=node_budget)
    survivors = info["survivors"]
    if len(survivors) == 1:
        value = survivors[0]
    elif model_value in survivors:
        value = model_value
    elif survivors:
        value = survivors[0]
    else:
        value = model_value  # poisoned board: the filter cannot help
    return {"value": value, "override": value != model_value, "search": info}


"""Build the Kaggle training views, including branch-focused augmentation.

Two changes relative to the original Modal baseline:

1. **Every branch decision is trained in both menu orders.**  The evaluator
   sorts the legal menu, so a model can learn "pick a001" instead of reading the
   board.  Reversing the two branch entries removes that shortcut.
2. **Wrong-branch trace rows are reused as corrected positives.**  The original
   `split_examples` drops rows with ``expert_policy_label: false``.  Their state
   is still a valid pre-branch state, and the audited solution tells us the
   correct value.  Adding that corrected example gives the model the hard states
   where the teacher probe guessed wrong, without ever training a wrong target.

Branch examples are optionally repeated so they are not drowned by the ~8.6k
deduction examples.
"""

import argparse
import json
import random
from collections import Counter
from pathlib import Path

from tt_core import (choice_id, completion, development_ids,
                           legal_actions, prompt, split_examples)

HERE = Path(__file__).resolve().parent
TILES = ("XX", "NS", "EW", "NE", "NW", "SE", "SW")


def load_jsonl(path: Path):
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle]


def load_solutions(path: Path) -> dict[str, list[list[str]]]:
    solutions = {}
    for record in load_jsonl(path):
        solutions[record["puzzle_id"]] = [
            [tile or "XX" for tile in row] for row in record["solution"]
        ]
    return solutions


def _make_example(row, actions, target):
    return {
        "user": prompt(row["state"], actions),
        "completion": completion(target),
        "kind": row["action"]["type"],
        "puzzle_id": row["puzzle_id"],
        "target": target,
        "menu_size": len(actions),
    }


def _branch_pair(mask: int) -> str:
    return "|".join(name for index, name in enumerate(TILES) if mask & (1 << index))


def _corrected_branch_row(row, solution):
    cell = row["action"]["cell"]
    correct = solution[cell["row"] - 1][cell["column"] - 1]
    actions = legal_actions(row["state"], pending=False)
    branch_actions = [a for a in actions if a["type"] == "branch"]
    if not any(a["cell"] == cell and a["value"] == correct for a in branch_actions):
        return None, None
    corrected = {"type": "branch", "cell": cell, "value": correct}
    corrected_row = dict(row)
    corrected_row["action"] = corrected
    return corrected_row, actions


def build_examples(rows, solutions=None, *, branch_repeat: int = 2,
                   reverse_branch_orders: bool = True,
                   corrected_wrong_branches: bool = True,
                   max_deductions_per_puzzle: int = 48,
                   seed: int = 42):
    """Return ``(examples, stats)`` for a list of trace rows."""
    selected = split_examples(rows, max_deductions_per_puzzle)
    examples = []
    by_kind = Counter()
    pairs = Counter()
    values = Counter()
    corrected = 0
    reversed_orders = 0

    def add(row, actions, target):
        examples.append(_make_example(row, actions, target))

    for row in selected:
        pending = row["action"]["type"] == "undo"
        actions = legal_actions(row["state"], pending=pending)
        target = choice_id(row["action"], actions)
        add(row, actions, target)
        by_kind[row["action"]["type"]] += 1
        if row["action"]["type"] == "branch":
            cell = row["action"]["cell"]
            pairs[_branch_pair(row["state"]["domains"][cell["row"] - 1][cell["column"] - 1])] += 1
            values[row["action"]["value"]] += 1
            if reverse_branch_orders:
                rev = list(reversed(actions))
                add(row, rev, choice_id(row["action"], rev))
                reversed_orders += 1

    if corrected_wrong_branches and solutions is not None:
        for row in rows:
            if row["action"]["type"] != "branch" or row.get("expert_policy_label", True):
                continue
            solution = solutions.get(row["puzzle_id"])
            if solution is None:
                continue
            corrected_row, actions = _corrected_branch_row(row, solution)
            if corrected_row is None:
                continue
            target = choice_id(corrected_row["action"], actions)
            add(corrected_row, actions, target)
            by_kind["branch_corrected"] += 1
            cell = corrected_row["action"]["cell"]
            pairs[_branch_pair(corrected_row["state"]["domains"][cell["row"] - 1][cell["column"] - 1])] += 1
            values[corrected_row["action"]["value"]] += 1
            corrected += 1
            if reverse_branch_orders:
                rev = list(reversed(actions))
                add(corrected_row, rev, choice_id(corrected_row["action"], rev))
                reversed_orders += 1

    if branch_repeat > 1:
        branch_examples = [
            e for e in examples if e["kind"] in {"branch", "branch_corrected"}
        ]
        for _ in range(branch_repeat - 1):
            examples.extend(branch_examples)

    rng = random.Random(seed)
    rng.shuffle(examples)
    stats = {
        "total": len(examples),
        "base_selected": len(selected),
        "by_kind": dict(by_kind),
        "corrected_wrong_branches": corrected,
        "reversed_branch_orders": reversed_orders,
        "branch_repeat": branch_repeat,
        "branch_examples_after_repeat": sum(
            1 for e in examples if e["kind"] in {"branch", "branch_corrected"}),
        "branch_pair_counts": dict(pairs),
        "branch_value_counts": dict(values),
    }
    return examples, stats


def build_dev_examples(rows, max_deductions_per_puzzle: int = 48):
    """Clean dev view: base examples only, no augmentation."""
    selected = split_examples(rows, max_deductions_per_puzzle)
    examples = []
    for row in selected:
        actions = legal_actions(row["state"], pending=row["action"]["type"] == "undo")
        target = choice_id(row["action"], actions)
        examples.append(_make_example(row, actions, target))
    return examples


def split_rows(rows):
    dev_ids = development_ids(row["puzzle_id"] for row in rows)
    train_rows = [row for row in rows if row["puzzle_id"] not in dev_ids]
    dev_rows = [row for row in rows if row["puzzle_id"] in dev_ids]
    return train_rows, dev_rows, len(dev_ids)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data", type=Path, default=HERE / "data/train.jsonl")
    parser.add_argument("--source", type=Path,
                        default=HERE / "data/train-puzzles-small.jsonl")
    parser.add_argument("--branch-repeat", type=int, default=2)
    parser.add_argument("--no-branch-augment", action="store_true")
    parser.add_argument("--write", type=Path, default=None,
                        help="Optional JSONL output with user/completion fields")
    args = parser.parse_args()

    rows = load_jsonl(args.data)
    solutions = load_solutions(args.source)
    train_rows, dev_rows, dev_count = split_rows(rows)
    if args.no_branch_augment:
        examples, stats = build_examples(
            train_rows, solutions, branch_repeat=1,
            reverse_branch_orders=False, corrected_wrong_branches=False)
    else:
        examples, stats = build_examples(
            train_rows, solutions, branch_repeat=args.branch_repeat)
    dev = build_dev_examples(dev_rows)
    stats["puzzles_total"] = len({r["puzzle_id"] for r in rows})
    stats["puzzles_train"] = len({r["puzzle_id"] for r in train_rows})
    stats["puzzles_dev"] = dev_count
    stats["dev_examples"] = len(dev)
    print(json.dumps(stats, indent=2))
    if args.write:
        with args.write.open("w", encoding="utf-8", newline="\n") as handle:
            for example in examples:
                handle.write(json.dumps(example, ensure_ascii=False) + "\n")
        print(f"wrote {len(examples)} examples to {args.write}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/tt/tt_eval.py
# Generated from the verified Kaggle package. Edit this cell, not a file.
"""Evaluate a trained adapter on Kaggle, with an optional safe branch search.

Zero-shot base model (no adapter), for a like-for-like comparison:

    python evaluate_kaggle.py --base-model --dataset small \
        --branch-search-depth 0 --output baseline.json

Trained adapter, model alone:

    python evaluate_kaggle.py --adapter /kaggle/working/run-branch/adapter \
        --branch-search-depth 0 --output baseline.json

Trained adapter with the depth-2 branch filter:

    python evaluate_kaggle.py --adapter /kaggle/working/run-branch/adapter \
        --branch-search-depth 2 --output improved.json

The search only rejects a value when propagation proves it impossible, so it
cannot reject the true value on a board that still contains the solution.  When
both values survive, the model's choice is kept.
"""

from __future__ import annotations

import argparse
import hashlib
import json
import sys
import time
from collections import Counter, defaultdict
from pathlib import Path

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))

from tt_core import (SYSTEM, development_ids, legal_actions,  # noqa: E402
                           parse_action, prompt, validate_action)
from tt_lib import DEFAULT_DEPTH, choose_branch  # noqa: E402
from tt_core import (BIT, clue_from_record, completed_error,  # noqa: E402
                         initial_domains, one, snapshot)


def parse_args():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--adapter", type=Path, default=None,
                        help="LoRA adapter directory; omit with --base-model")
    parser.add_argument("--base-model", action="store_true",
                        help="Evaluate the base checkpoint with no adapter (zero-shot)")
    parser.add_argument("--data", type=Path,
                        default=HERE / "data/train-puzzles-medium.jsonl")
    parser.add_argument("--small-data", type=Path,
                        default=HERE / "data/train-puzzles-small.jsonl")
    parser.add_argument("--dataset", choices=("medium", "small"), default="medium")
    parser.add_argument("--model", default="Qwen/Qwen2.5-1.5B-Instruct")
    parser.add_argument("--limit", type=int, default=32,
                        help="0 means all; medium uses a shape-stratified sample")
    parser.add_argument("--max-actions", type=int, default=256)
    parser.add_argument("--max-new-tokens", type=int, default=32)
    parser.add_argument("--branch-search-depth", type=int, default=DEFAULT_DEPTH,
                        help="0 disables the branch filter")
    parser.add_argument("--branch-search-budget", type=int, default=4000,
                        help="Max search nodes per branch decision")
    parser.add_argument("--output", type=Path, default=Path("/kaggle/working/eval.json"))
    parser.add_argument("--only", default=None,
                        help="Comma-separated puzzle ids to evaluate; bypasses sampling")
    parser.add_argument("--compact", action="store_true",
                        help="Drop board snapshots from the saved history")
    parser.add_argument("--local-files-only", action="store_true")
    args = parser.parse_args()
    if args.adapter is None and not args.base_model:
        parser.error("give --adapter for a trained policy, or --base-model for zero-shot")
    if args.adapter is not None and args.base_model:
        parser.error("--adapter and --base-model are mutually exclusive")
    return args


def state_for(board, clue, stack):
    return {
        "row_clues": clue["rows"], "column_clues": clue["columns"],
        "givens": [{"row": r + 1, "column": c + 1, "value": shape}
                   for r, c, shape in clue["given_shapes"]],
        "domains": [line.copy() for line in board],
        "branch_stack": [{"to_step": item["to_step"], "cell": item["cell"],
                          "value": item["value"]} for item in stack],
    }


def select_medium_sample(puzzles, limit):
    groups = defaultdict(list)
    for puzzle in puzzles:
        shape = (len(puzzle["puzzle"]["rows"]), len(puzzle["puzzle"]["columns"]))
        groups[shape].append(puzzle)
    for group in groups.values():
        group.sort(key=lambda row: hashlib.sha256(row["puzzle_id"].encode()).hexdigest())
    target = min(limit, sum(len(group) for group in groups.values()))
    selected = {shape: [] for shape in groups}
    while sum(len(items) for items in selected.values()) < target:
        eligible = [shape for shape in groups if len(selected[shape]) < len(groups[shape])]
        shape = min(eligible, key=lambda key: (len(selected[key]), key))
        selected[shape].append(groups[shape][len(selected[shape])])
    return sorted((p for items in selected.values() for p in items),
                  key=lambda row: row["puzzle_id"])


def model_policy(model, tokenizer, args):
    """Return ``policy(state, actions) -> raw completion text`` for the adapter."""
    import torch

    def policy(state, actions):
        messages = [{"role": "system", "content": SYSTEM},
                    {"role": "user", "content": prompt(state, actions)}]
        encoded = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_dict=True,
            return_tensors="pt").to("cuda")
        with torch.inference_mode():
            generated = model.generate(
                **encoded, max_new_tokens=args.max_new_tokens, do_sample=False,
                pad_token_id=tokenizer.eos_token_id)
        return tokenizer.decode(generated[0, encoded["input_ids"].shape[-1]:],
                                skip_special_tokens=True)

    return policy


def rollout(policy, record, args):
    """Run one puzzle with a ``policy(state, actions) -> raw text`` callable.

    The policy seam keeps the harness testable without a GPU: ``selfcheck.py``
    drives the exact same board updates, branch stack and undo path with an
    oracle policy that reads the audited solution.
    """
    clue = clue_from_record(record)
    board = initial_domains(clue)
    stack, history = [], []
    pending = False
    failure = None
    started = time.monotonic()
    for step_index in range(args.max_actions):
        state = state_for(board, clue, stack)
        actions = legal_actions(state, pending=pending)
        if not actions:
            failure = "no_legal_actions"
            break
        raw = policy(state, actions)
        action = parse_action(raw, actions)
        if action is None:
            valid, reason, canonical = False, "invalid_choice", None
        else:
            valid, reason, canonical = validate_action(board, clue, action, stack, pending)

        model_action = canonical
        search = None
        if valid and canonical["type"] == "branch" and args.branch_search_depth > 0:
            decision = choose_branch(
                state, canonical["cell"], canonical["value"],
                depth=args.branch_search_depth,
                node_budget=args.branch_search_budget)
            search = {"model_value": canonical["value"],
                      "chosen_value": decision["value"],
                      "override": decision["override"],
                      "survivors": decision["search"]["survivors"] if decision["search"] else None,
                      "feasible": decision["search"]["feasible"] if decision["search"] else None,
                      "depth": args.branch_search_depth}
            if decision["override"]:
                candidate = {"type": "branch", "cell": canonical["cell"],
                             "value": decision["value"]}
                valid, reason, canonical = validate_action(board, clue, candidate, stack, pending)
                if not valid:  # should not happen; keep the model choice
                    canonical = model_action
                    search["override"] = False

        event = {"step": step_index, "state": state, "raw": raw,
                 "action": canonical if valid else action,
                 "model_action": model_action,
                 "branch_search": search,
                 "legal_action_count": len(actions), "valid": valid, "reason": reason}
        if args.compact:
            event.pop("state", None)
        history.append(event)
        if not valid:
            failure = reason
            break
        kind = canonical["type"]
        if kind == "deduce":
            cell = canonical["cell"]
            r, c = cell["row"] - 1, cell["column"] - 1
            board[r][c] = sum(BIT[x] for x in canonical["candidates"])
        elif kind == "branch":
            cell = canonical["cell"]
            r, c = cell["row"] - 1, cell["column"] - 1
            stack.append({"to_step": step_index, "cell": cell,
                          "value": canonical["value"], "checkpoint": snapshot(board)})
            board[r][c] = BIT[canonical["value"]]
        elif kind == "contradiction":
            if not stack:
                failure = "contradiction_without_branch"
                break
            pending = True
        elif kind == "undo":
            branch = stack.pop()
            board = snapshot(branch["checkpoint"])
            r, c = branch["cell"]["row"] - 1, branch["cell"]["column"] - 1
            board[r][c] &= ~BIT[branch["value"]]
            pending = False
        elif kind == "finish":
            break
    else:
        failure = "action_limit"

    grid = [[one(mask) for mask in row] for row in board]
    reference = [[tile or "XX" for tile in row] for row in record["solution"]]
    correct = sum(a == b for row_a, row_b in zip(grid, reference)
                  for a, b in zip(row_a, row_b))
    finished = bool(history and history[-1]["valid"]
                    and history[-1]["action"]["type"] == "finish")
    return {
        "puzzle_id": record["puzzle_id"],
        "solved": finished and grid == reference,
        "valid_complete_board": finished and completed_error(board, clue) is None,
        "finished": finished, "failure": failure,
        "correct_cells": correct,
        "total_cells": len(reference) * len(reference[0]),
        "actions": len(history),
        "search_overrides": sum(1 for event in history
                                if event.get("branch_search")
                                and event["branch_search"]["override"]),
        "seconds": round(time.monotonic() - started, 3),
        "prediction": grid, "reference": reference, "history": history,
    }


def filter_only(puzzles, spec):
    """Restrict a puzzle list to the ids in ``spec`` (comma separated)."""
    wanted = [part.strip() for part in spec.split(",") if part.strip()]
    by_id = {record["puzzle_id"]: record for record in puzzles}
    missing = [pid for pid in wanted if pid not in by_id]
    if missing:
        raise SystemExit(f"--only: not in this split/sample: {missing}")
    return [by_id[pid] for pid in wanted]


def load_policy(args):
    """Load the adapter (or the bare base model) and return the policy.

    Shared by the evaluator and the DAgger collector so both see the identical
    tokenizer, dtype and chat template.  With ``--base-model`` no adapter is
    loaded, which is the zero-shot baseline.
    """
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    base_only = args.base_model or args.adapter is None
    tokenizer = None
    if not base_only:
        try:
            tokenizer = AutoTokenizer.from_pretrained(
                args.adapter, local_files_only=args.local_files_only)
        except (OSError, ValueError):
            tokenizer = None
    if tokenizer is None:
        # Older adapters may only contain weights/config; the base tokenizer is
        # the same one training used.
        tokenizer = AutoTokenizer.from_pretrained(
            args.model, local_files_only=args.local_files_only)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    # bf16 requires sm_80+.  `is_bf16_supported()` can return True on some T4
    # images, where bf16 then falls back to slow emulation.
    capability = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
    bf16_ok = capability[0] >= 8 and torch.cuda.is_bf16_supported()
    dtype = torch.bfloat16 if bf16_ok else torch.float16
    base = AutoModelForCausalLM.from_pretrained(
        args.model, dtype=dtype, attn_implementation="sdpa",
        local_files_only=args.local_files_only)
    if base_only:
        model = base
    else:
        from peft import PeftModel
        model = PeftModel.from_pretrained(base, str(args.adapter))
    model = model.to("cuda").eval()
    model.config.use_cache = True
    return model_policy(model, tokenizer, args)


def run_label(args):
    """Identifier for the report: adapter name, or the base model."""
    if args.base_model or args.adapter is None:
        return "baseline-" + args.model.split("/")[-1].lower()
    return args.adapter.parent.name if args.adapter.name == "adapter" else args.adapter.name


def main():
    args = parse_args()

    if args.dataset == "medium":
        source = [json.loads(line) for line in args.data.open(encoding="utf-8")]
        puzzles = [record for record in source if record.get("split") == "test_medium"]
        if not puzzles:
            raise SystemExit("No test_medium puzzles in the data file")
        split = "test_medium"
        sample_strategy = "full_set"
        if args.only:
            puzzles = filter_only(puzzles, args.only)
            sample_strategy = "explicit_ids"
        elif args.limit:
            puzzles = select_medium_sample(puzzles, args.limit)
            sample_strategy = "deterministic_board_size_stratified"
    else:
        source = [json.loads(line) for line in args.small_data.open(encoding="utf-8")]
        dev_ids = development_ids(record["puzzle_id"] for record in source)
        puzzles = [record for record in source if record["puzzle_id"] in dev_ids]
        puzzles.sort(key=lambda record: record["puzzle_id"])
        split = "internal_puzzle_holdout_from_212_training_puzzles"
        sample_strategy = "dev_holdout"
        if args.only:
            puzzles = filter_only(puzzles, args.only)
            sample_strategy = "explicit_ids"
        elif args.limit:
            puzzles = puzzles[:args.limit]

    results = []
    policy = load_policy(args)
    for index, record in enumerate(puzzles, 1):
        result = rollout(policy, record, args)
        results.append(result)
        print(f"{args.dataset} evaluation: {index}/{len(puzzles)} "
              f"({result['puzzle_id']}: {'solved' if result['solved'] else result['failure']})",
              flush=True)

    report = {
        "run_id": run_label(args),
        "dataset": args.dataset,
        "split": split,
        "sample_strategy": sample_strategy,
        "branch_search_depth": args.branch_search_depth,
        "attempted": len(results),
        "solved": sum(r["solved"] for r in results),
        "valid_complete_boards": sum(r["valid_complete_board"] for r in results),
        "correct_cells": sum(r["correct_cells"] for r in results),
        "total_cells": sum(r["total_cells"] for r in results),
        "failures": {reason: sum(r["failure"] == reason for r in results)
                     for reason in sorted({r["failure"] for r in results if r["failure"]})},
        "search_overrides": sum(r["search_overrides"] for r in results),
        "puzzles": results,
    }
    args.output.parent.mkdir(parents=True, exist_ok=True)
    args.output.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
    summary = {key: value for key, value in report.items() if key != "puzzles"}
    summary["output"] = str(args.output)
    print(json.dumps(summary, indent=2))


if __name__ == "__main__":
    main()


## 5. Checks

`run()` streams and logs every command; the two CLI validations matter because
`--adapter` is no longer required — a typo could otherwise evaluate the base model
while you think you are measuring run-04.


In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

LOGS = Path("/kaggle/working/logs")
LOGS.mkdir(parents=True, exist_ok=True)


def run(cmd, name, extra_env=None):
    """Run a command, stream and log its output, raise on non-zero exit."""
    cmd = [str(part) for part in cmd]
    print("$", " ".join(cmd), flush=True)
    log_path = LOGS / f"{name}.log"
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "TOKENIZERS_PARALLELISM": "false"}
    if extra_env:
        env.update(extra_env)
    with log_path.open("w", encoding="utf-8") as log:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, env=env, bufsize=1)
        for line in proc.stdout:
            print(line, end="")
            log.write(line)
    if proc.wait():
        raise RuntimeError(f"{name} failed; full output in {log_path}")
    print(f"[{name} ok] log: {log_path}", flush=True)


import torch

print("python     :", sys.executable)
print("accelerate :", shutil.which("accelerate") or "NOT FOUND")
print("torch      :", torch.__version__, "| cuda", torch.version.cuda,
      "| available", torch.cuda.is_available(), "| gpus", torch.cuda.device_count())
assert torch.cuda.is_available(), "torch cannot see a GPU; check the accelerator setting and the pip step"
assert torch.cuda.device_count() >= 1, "no CUDA devices"

LAUNCH = [sys.executable, "-m", "accelerate.commands.launch",
          "--multi_gpu", "--num_processes", "2",
          "--mixed_precision", "fp16", "--main_process_port", "29501"]
print("launcher   :", " ".join(LAUNCH[:3]), "..." if torch.cuda.device_count() >= 2
      else "(only one GPU: DDP steps will be skipped)")


In [ ]:
%cd /kaggle/working/tt
run([sys.executable, "-m", "py_compile", "tt_core.py", "tt_lib.py", "tt_eval.py"], "compile")
run([sys.executable, "-c", "import importlib.util; found = importlib.util.find_spec('torchao') is not None; print('torchao installed:', found); raise SystemExit(1 if found else 0)"], "torchao-check")
run([sys.executable, "-c", "import tt_core, tt_lib, tt_eval; print('module imports ok')"], "imports")
run([sys.executable, "tt_eval.py", "--help"], "eval-help")
run([sys.executable, "-c", "import subprocess, sys; p = subprocess.run([sys.executable, 'tt_eval.py'], capture_output=True, text=True); print(p.returncode, p.stderr.strip().splitlines()[-1]); raise SystemExit(0 if p.returncode != 0 and 'give --adapter' in p.stderr else 1)"], "require-adapter-or-base")


### Cache the base weights once

The base arms and the adapter arms both start from the same ~3.1 GB checkpoint. Fetch
it here so four processes do not fight over the HF cache lock.


In [ ]:
%cd /kaggle/working/tt
run([sys.executable, "-c",
     "from huggingface_hub import snapshot_download; "
     f"snapshot_download('{MODEL}'); print('cached {MODEL}')"], "model-cache")


## 6. Confirm the 32-puzzle holdout

The evaluator's `--dataset small` path uses `development_ids`, a deterministic
puzzle-id hash over the 212 small puzzles. Printing and asserting the 32 ids here means
the four arms cannot silently evaluate different sets, and a data-file change shows up
before any GPU time is spent.

Expected: **32 puzzles, 1762 cells**, shapes 6x6 through 9x9.


In [ ]:
%cd /kaggle/working/tt
import json
import tt_core

small = [json.loads(line) for line in
         (DATA / "train-puzzles-small.jsonl").open(encoding="utf-8")]
DEV_IDS = sorted(tt_core.development_ids(record["puzzle_id"] for record in small))
by_id = {record["puzzle_id"]: record for record in small}
cells = sum(len(by_id[pid]["solution"]) * len(by_id[pid]["solution"][0]) for pid in DEV_IDS)
shapes = {}
for pid in DEV_IDS:
    record = by_id[pid]
    shape = (len(record["puzzle"]["rows"]), len(record["puzzle"]["columns"]))
    shapes[shape] = shapes.get(shape, 0) + 1
print(f"holdout: {len(DEV_IDS)} puzzles | {cells} cells")
print("shapes:", dict(sorted(shapes.items())))
assert len(DEV_IDS) == 32, len(DEV_IDS)
assert cells == 1762, cells
print("ids:", DEV_IDS)


## 7. Run the four arms

Two arms per wave, one per GPU, so wall time is two waves. Each arm evaluates all 32
ids with `--only`, which also marks the report `sample_strategy: explicit_ids` — a
partial run can never be mistaken for the full set.

Settings are identical across arms: **256 actions, 32 new tokens, greedy**. Only the
weights (`--base-model` vs `--adapter`) and `--branch-search-depth` change.

If only one GPU is visible the arms run sequentially; the cell says so instead of
failing.


In [ ]:
%cd /kaggle/working/tt
from concurrent.futures import ThreadPoolExecutor

ARMS = [
    {"tag": "baseline-depth0", "base": True, "depth": 0},
    {"tag": "baseline-depth2", "base": True, "depth": 2},
    {"tag": "run04-depth0", "base": False, "depth": 0},
    {"tag": "run04-depth2", "base": False, "depth": 2},
]
WAVES = [ARMS[0:2], ARMS[2:4]]


def run_arm(arm, device):
    output = Path(f"/kaggle/working/small-{arm['tag']}.json")
    if output.exists():
        done = json.loads(output.read_text(encoding="utf-8")).get("attempted")
        if done == len(DEV_IDS):
            print(f"[{arm['tag']}] already complete, skipping")
            return 0
    cmd = [sys.executable, "tt_eval.py",
           "--dataset", "small",
           "--small-data", str(DATA / "train-puzzles-small.jsonl"),
           "--model", MODEL,
           "--max-actions", "256",
           "--max-new-tokens", "32",
           "--branch-search-depth", str(arm["depth"]),
           "--only", ",".join(DEV_IDS),
           "--output", str(output),
           "--compact"]
    cmd += ["--base-model"] if arm["base"] else ["--adapter", str(ADAPTER)]
    log_path = LOGS / f"small-{arm['tag']}.log"
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "TOKENIZERS_PARALLELISM": "false",
           "CUDA_VISIBLE_DEVICES": device}
    print(f"[{arm['tag']}] GPU {device} -> {log_path}", flush=True)
    with log_path.open("w", encoding="utf-8") as log:
        proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT,
                                text=True, env=env)
        status = proc.wait()
    print(f"[{arm['tag']}] exit {status}", flush=True)
    return status


failed = []
for wave_index, wave in enumerate(WAVES):
    if torch.cuda.device_count() >= len(wave):
        with ThreadPoolExecutor(max_workers=len(wave)) as pool:
            statuses = list(pool.map(lambda pair: run_arm(pair[1], str(pair[0])),
                                     list(enumerate(wave))))
    else:
        print(f"one GPU visible: wave {wave_index} runs sequentially")
        statuses = [run_arm(arm, "0") for arm in wave]
    failed.extend(arm["tag"] for arm, status in zip(wave, statuses) if status)

for tag in failed:
    tail = (LOGS / f"small-{tag}.log").read_text(encoding="utf-8").splitlines()[-20:]
    print(f"\n--- {tag} tail ---")
    print("\n".join(tail))
if len(failed) == len(ARMS):
    raise RuntimeError("all arms failed; see /kaggle/working/logs/small-*.log")
print("arms failed:", failed)


## 8. Compare

Read the summary first, then the per-puzzle table.

* **`invalid`** is the format check. The base model has never been trained on
  `{{"choice": "aNNN"}}`, so a large `invalid` count is the expected result — it means
  the failure is instruction/format following, not puzzle reasoning.
* **`solved`** is the headline, but `cells` shows how close a failure was: a base arm
  can be at 3/49 cells and a run-04 arm at 48/49, and both count as unsolved.
* **`overrides`** is how often the depth-2 filter replaced a branch value. On the base
  arms it should be near zero: the model rarely reaches a branch with a parseable
  choice.

Recorded reference (run-03 adapter, depth 0, 256 actions): **31/32, 1733/1762 cells,
0 invalid**. Run-04 is a different adapter, so treat it as context, not a target.


In [ ]:
import json
from pathlib import Path
import pandas as pd

ARMS = ["baseline-depth0", "baseline-depth2", "run04-depth0", "run04-depth2"]
LABEL = {
    "baseline-depth0": "base, no filter",
    "baseline-depth2": "base, depth 2",
    "run04-depth0": "run-04, no filter",
    "run04-depth2": "run-04, depth 2",
}

reports = {}
for tag in ARMS:
    path = Path(f"/kaggle/working/small-{tag}.json")
    assert path.exists(), f"missing {path.name}: re-run section 7"
    reports[tag] = json.loads(path.read_text(encoding="utf-8"))

summary = []
for tag in ARMS:
    report = reports[tag]
    puzzles = report["puzzles"]
    invalid = sum(1 for p in puzzles for e in (p.get("history") or []) if not e.get("valid"))
    actions = sorted(p["actions"] for p in puzzles)
    summary.append({
        "arm": tag,
        "what": LABEL[tag],
        "solved": f"{report['solved']}/{report['attempted']}",
        "cells": f"{report['correct_cells']}/{report['total_cells']}",
        "invalid": invalid,
        "median_actions": actions[len(actions) // 2],
        "overrides": report["search_overrides"],
        "failures": report["failures"],
    })
display(pd.DataFrame(summary))

def solved_set(tag):
    return {p["puzzle_id"] for p in reports[tag]["puzzles"] if p["solved"]}

base0, base2 = solved_set("baseline-depth0"), solved_set("baseline-depth2")
r04_0, r04_2 = solved_set("run04-depth0"), solved_set("run04-depth2")

print()
print(f"adapter effect, no filter : {len(base0)}/32 -> {len(r04_0)}/32")
print(f"adapter effect, depth 2   : {len(base2)}/32 -> {len(r04_2)}/32")
print(f"filter effect on base     : {len(base0)}/32 -> {len(base2)}/32")
print(f"filter effect on run-04   : {len(r04_0)}/32 -> {len(r04_2)}/32")
print(f"gained by adapter (no filter): {sorted(p.replace('-small','') for p in r04_0 - base0)}")
print(f"gained by adapter (depth 2)  : {sorted(p.replace('-small','') for p in r04_2 - base2)}")

rows = []
for index, puzzle in enumerate(reports["run04-depth2"]["puzzles"]):
    pid = puzzle["puzzle_id"]
    row = {"puzzle": pid.replace("-small", "")}
    for tag in ARMS:
        entry = next(p for p in reports[tag]["puzzles"] if p["puzzle_id"] == pid)
        mark = "Y" if entry["solved"] else "n"
        row[tag] = f"{mark} {entry['correct_cells']}/{entry['total_cells']}"
    rows.append(row)
display(pd.DataFrame(rows))

merged = {
    "comparison": "small-dev-baseline-vs-run04",
    "model": MODEL,
    "max_actions": 256,
    "max_new_tokens": 32,
    "arms": {tag: {key: reports[tag][key] for key in
                   ("run_id", "attempted", "solved", "correct_cells", "total_cells",
                    "search_overrides", "failures")} for tag in ARMS},
    "puzzles": [
        {"puzzle_id": p["puzzle_id"],
         "arms": {tag: {key: next(q for q in reports[tag]["puzzles"]
                                 if q["puzzle_id"] == p["puzzle_id"])[key]
                        for key in ("solved", "correct_cells", "total_cells", "actions",
                                    "failure")}
                  for tag in ARMS}}
        for p in reports["run04-depth2"]["puzzles"]
    ],
}
Path("/kaggle/working/small-compare.json").write_text(
    json.dumps(merged, indent=1), encoding="utf-8")
print("wrote /kaggle/working/small-compare.json")


## 9. Save the artifacts

Four arm JSONs (with per-action histories), the merged comparison, and the logs.


In [ ]:
import pathlib, shutil

art = pathlib.Path("/kaggle/working/artifacts")
art.mkdir(exist_ok=True)
for pattern in ("small-*.json", "small-compare.json"):
    for path in pathlib.Path("/kaggle/working").glob(pattern):
        shutil.copy(path, art / path.name)
if LOGS.exists():
    shutil.copytree(LOGS, art / "logs", dirs_exist_ok=True)
print(sorted(p.name for p in art.iterdir()))
print("download the contents of", art)
